# Neurosynth — Masks & Decoding

Self-contained notebook: all paths are defined once in **Configuration** below.  
Run top-to-bottom; no variables needed from other notebooks.

| Section | What it does | Output location |
|---------|-------------|----------------|
| **A** | ISPC masks — one per condition (4 conditions, FDR-sig + positive) | `parcel_ispc/leftwing/neurosynth/` |
| **B** | IS-RSA warmth masks — one per condition × model (8 combos, FDR-sig + positive) | `rsa/parcel_isrsa_leftwing/warmth_isrsa/neurosynth/` |
| **C** | ISPC contrast masks — one per contrast (3 contrasts, FDR-sig, both directions) | `parcel_ispc/leftwing/contrasts/neurosynth/` |
| **D** | Decoder results — bar charts for contrast JSON files from neurosynth.org | *(display only)* |

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from nilearn.image import resample_to_img
from IPython.display import display

## Configuration

The only cell you need to edit if paths change.

In [ ]:
ROOT = Path('/path/to/project')

# ── Atlases ────────────────────────────────────────────────────────────────────
ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'
AAL2_NII   = ROOT / 'data/atlases/aal2_for_SPM12/aal/aal2.nii.gz'
AAL2_TXT   = ROOT / 'data/atlases/aal2_for_SPM12/aal/aal2.nii.txt'

# ── Section A: ISPC per-condition ──────────────────────────────────────────────
ISPC_DIR         = ROOT / 'data/derivatives/parcel_ispc/leftwing'
ISPC_SIG_CSV     = ISPC_DIR / 'parcel_isc_B_significance.csv'
NS_ISPC_DIR      = ISPC_DIR / 'neurosynth'          # output

# ── Section B: IS-RSA warmth ───────────────────────────────────────────────────
WARMTH_DIR       = ROOT / 'data/derivatives/rsa/parcel_isrsa_leftwing/warmth_isrsa'
NS_WARMTH_DIR    = WARMTH_DIR / 'neurosynth'         # output

# ── Section C: ISPC contrasts ──────────────────────────────────────────────────
CONTRAST_DIR     = ISPC_DIR / 'contrasts'
NS_CONTRAST_DIR  = CONTRAST_DIR / 'neurosynth'       # output

# ── Section D: Decoder results ─────────────────────────────────────────────────
DECODER_DIR      = ROOT / 'data/neurosynth'          # JSON files from neurosynth.org

# ── Analysis parameters ────────────────────────────────────────────────────────
CONDITIONS  = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
MODELS      = ['inparty', 'outparty']
FDR_Q       = 0.05
TOP_N       = 25    # top/bottom N terms to show in decoder bar charts

# Create output directories
for d in [NS_ISPC_DIR, NS_WARMTH_DIR, NS_CONTRAST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuration OK')

## Atlas Setup & Shared Helpers

Loads the Schaefer+Tian parcellation atlas and the AAL2 atlas once.  
All mask-creation sections below use the functions defined here.

In [ ]:
# ── Schaefer + Tian atlas ──────────────────────────────────────────────────────
_atlas_img    = nib.load(ATLAS_NII)
_atlas_data   = np.asarray(_atlas_img.dataobj)
_atlas_affine = _atlas_img.affine

_labels_df  = pd.read_csv(LABELS_TSV, sep='\t')
_name_to_id = dict(zip(_labels_df['name'], _labels_df['id']))

print(f'Atlas: {_atlas_data.shape}  |  {len(_name_to_id)} labelled parcels')

# ── AAL2 (anatomical region labels) ───────────────────────────────────────────
_aal_rs   = resample_to_img(nib.load(AAL2_NII), _atlas_img,
                             interpolation='nearest',
                             force_resample=True, copy_header=True)
_aal_data = np.asarray(_aal_rs.dataobj).astype(int)

_aal_labels = {}
with open(AAL2_TXT) as _fh:
    for _line in _fh:
        _p = _line.strip().split()
        if len(_p) >= 2:
            _aal_labels[int(_p[0])] = _p[1]

print(f'AAL2: {len(_aal_labels)} regions')

# ── Parcel utilities ───────────────────────────────────────────────────────────
def _parse_parcel(name):
    """Return (hemisphere, network, subregion) from a 7Networks parcel name."""
    parts = name.split('_')
    hemi      = parts[1]
    net       = parts[2]
    sub_parts = parts[3:]
    subregion = '_'.join(sub_parts[:-1]) if len(sub_parts) > 1 else ''
    return hemi, net, subregion or net

def _mni_centroid(pid):
    mask = _atlas_data == pid
    if not mask.any():
        return np.full(3, np.nan)
    vox = np.array(ndimage.center_of_mass(mask))
    return np.round(nib.affines.apply_affine(_atlas_affine, vox), 1)

def _top_aal(pid):
    mask = _atlas_data == pid
    vals = _aal_data[mask]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return 'no_overlap'
    uniq, counts = np.unique(vals, return_counts=True)
    return _aal_labels.get(int(uniq[np.argmax(counts)]), 'unknown')

def _save_mask(parcel_ids, path):
    vol = np.zeros(_atlas_data.shape, dtype=np.uint8)
    for pid in parcel_ids:
        vol[_atlas_data == pid] = 1
    nib.save(nib.Nifti1Image(vol, _atlas_affine, _atlas_img.header), path)

def _save_effect_map(parcel_id_to_val, path):
    """Float map — each parcel's voxels filled with its effect-size value."""
    vol = np.zeros(_atlas_data.shape, dtype=np.float32)
    for pid, v in parcel_id_to_val.items():
        vol[_atlas_data == pid] = v
    nib.save(nib.Nifti1Image(vol, _atlas_affine, _atlas_img.header), path)

print('Helpers ready.')

---
## Section A — ISPC Masks (4 conditions)

Source: `parcel_isc_B_significance.csv`  
Criterion: FDR-significant (`p_fdr < 0.05`) **and** `isc_mean > 0`  
Effect map: `isc_mean` value per parcel voxel

```
parcel_ispc/leftwing/neurosynth/
  {condition}/
    mask_combined.nii.gz   ← upload to neurosynth.org/decode/
    ispc_map.nii.gz        ← isc_mean effect map
    parcels/               ← one mask per parcel
    parcel_locations.csv
  all_conditions_parcel_locations.csv
```

In [ ]:
sig_df = pd.read_csv(ISPC_SIG_CSV)
print(f'Loaded: {ISPC_SIG_CSV.name}  shape={sig_df.shape}')
print(f'Conditions: {sig_df.condition.unique().tolist()}\n')

ispc_all_rows = []

for cond in CONDITIONS:
    sub     = sig_df[sig_df['condition'] == cond]
    pos_sig = sub[(sub['significant'] == True) & (sub['isc_mean'] > 0)]

    print(f"{'='*62}")
    print(f"A — {cond}  |  {len(pos_sig)} FDR-sig positive parcels")
    print('='*62)

    out_dir     = NS_ISPC_DIR / cond
    parcels_dir = out_dir / 'parcels'
    out_dir.mkdir(exist_ok=True)
    parcels_dir.mkdir(exist_ok=True)

    rows, pids = [], []

    for _, row in pos_sig.iterrows():
        pname = row['parcel_name']
        pid   = _name_to_id.get(pname)
        if pid is None:
            print(f'  WARNING: {pname} not in atlas — skipped')
            continue

        pids.append(pid)
        hemi, net, sub_reg = _parse_parcel(pname)
        mni = _mni_centroid(pid)

        rows.append({
            'condition':   cond,
            'parcel_name': pname,
            'hemisphere':  hemi,
            'network':     net,
            'subregion':   sub_reg,
            'aal_region':  _top_aal(pid),
            'mni_x': mni[0], 'mni_y': mni[1], 'mni_z': mni[2],
            'isc_mean': round(float(row['isc_mean']), 4),
            'p_fdr':    round(float(row['p_fdr']), 5),
        })
        _save_mask([pid], parcels_dir / f'{pname}.nii.gz')

    if not pids:
        print('  No significant positive parcels — skipping mask creation.\n')
        continue

    _save_mask(pids, out_dir / 'mask_combined.nii.gz')
    _save_effect_map({_name_to_id[r['parcel_name']]: r['isc_mean'] for r in rows},
                     out_dir / 'ispc_map.nii.gz')

    cdf = pd.DataFrame(rows)
    cdf.to_csv(out_dir / 'parcel_locations.csv', index=False)

    print(f"  mask_combined.nii.gz  ({len(pids)} parcels)")
    print(f"  ispc_map.nii.gz")
    print(f"  parcel_locations.csv")
    print()
    display(
        cdf[['parcel_name', 'network', 'hemisphere', 'subregion',
             'aal_region', 'mni_x', 'mni_y', 'mni_z', 'isc_mean', 'p_fdr']]
        .sort_values('isc_mean', ascending=False)
        .reset_index(drop=True)
    )
    ispc_all_rows.extend(rows)

master = NS_ISPC_DIR / 'all_conditions_parcel_locations.csv'
pd.DataFrame(ispc_all_rows).to_csv(master, index=False)
print(f'Master CSV ({len(ispc_all_rows)} rows) → {master.relative_to(ROOT)}')

---
## Section B — IS-RSA Warmth Masks (4 conditions × 2 models)

Source: `warmth_isrsa/B_{Condition}_{model}.csv`  
Criterion: FDR-significant (`significant == True`) **and** `rsa_r > 0`  
Effect map: `rsa_r` value per parcel voxel  
Note: combos with zero significant positive parcels are skipped gracefully.

```
rsa/parcel_isrsa_leftwing/warmth_isrsa/neurosynth/
  {condition}_{model}/
    mask_combined.nii.gz   ← upload to neurosynth.org/decode/
    rsa_r_map.nii.gz       ← rsa_r effect map
    parcels/
    parcel_locations.csv
  all_warmth_parcel_locations.csv
```

In [ ]:
warmth_all_rows = []

for cond in CONDITIONS:
    for model in MODELS:
        csv_path = WARMTH_DIR / f'B_{cond}_{model}.csv'
        if not csv_path.exists():
            print(f'[MISSING] {csv_path.name}')
            continue

        df      = pd.read_csv(csv_path)
        pos_sig = df[(df['significant'] == True) & (df['rsa_r'] > 0)]
        label   = f'{cond}_{model}'

        print(f"{'='*62}")
        print(f"B — {label}  |  {len(pos_sig)} FDR-sig positive parcels")
        print('='*62)

        out_dir     = NS_WARMTH_DIR / label
        parcels_dir = out_dir / 'parcels'
        out_dir.mkdir(exist_ok=True)
        parcels_dir.mkdir(exist_ok=True)

        rows, pids = [], []

        for _, row in pos_sig.iterrows():
            pname = row['parcel']
            pid   = _name_to_id.get(pname)
            if pid is None:
                print(f'  WARNING: {pname} not in atlas — skipped')
                continue

            pids.append(pid)
            hemi, net, sub_reg = _parse_parcel(pname)
            mni = _mni_centroid(pid)

            rows.append({
                'condition':   cond,
                'model':       model,
                'parcel_name': pname,
                'hemisphere':  hemi,
                'network':     net,
                'subregion':   sub_reg,
                'aal_region':  _top_aal(pid),
                'mni_x': mni[0], 'mni_y': mni[1], 'mni_z': mni[2],
                'rsa_r': round(float(row['rsa_r']), 4),
                'p_fdr': round(float(row['p_fdr']), 5),
            })
            _save_mask([pid], parcels_dir / f'{pname}.nii.gz')

        if not pids:
            print('  No significant positive parcels — skipping mask creation.\n')
            continue

        _save_mask(pids, out_dir / 'mask_combined.nii.gz')
        _save_effect_map({_name_to_id[r['parcel_name']]: r['rsa_r'] for r in rows},
                         out_dir / 'rsa_r_map.nii.gz')

        cdf = pd.DataFrame(rows)
        cdf.to_csv(out_dir / 'parcel_locations.csv', index=False)

        print(f"  mask_combined.nii.gz  ({len(pids)} parcels)")
        print(f"  rsa_r_map.nii.gz")
        print(f"  parcel_locations.csv")
        print()
        display(
            cdf[['parcel_name', 'network', 'hemisphere', 'subregion',
                 'aal_region', 'mni_x', 'mni_y', 'mni_z', 'rsa_r', 'p_fdr']]
            .sort_values('rsa_r', ascending=False)
            .reset_index(drop=True)
        )
        warmth_all_rows.extend(rows)

master = NS_WARMTH_DIR / 'all_warmth_parcel_locations.csv'
pd.DataFrame(warmth_all_rows).to_csv(master, index=False)
print(f'Master CSV ({len(warmth_all_rows)} rows) → {master.relative_to(ROOT)}')

---
## Section C — ISPC Contrast Masks (3 contrasts)

Source: `parcel_ispc/leftwing/contrasts/contrast_*_subject_level.csv`  
Criterion: FDR-significant (`significant == True`), both directions included  
Effect map: Cohen's d per parcel voxel

```
parcel_ispc/leftwing/contrasts/neurosynth/
  proleft_vs_antiright/
    mask_combined.nii.gz   ← upload to neurosynth.org/decode/
    cohend_map.nii.gz      ← Cohen's d effect map
    parcels/
    parcel_locations.csv
  proright_vs_antileft/
  agreed_vs_disagreed/
  all_contrasts_parcel_locations.csv
```

In [ ]:
# Contrast definitions
# cond_A / cond_B: match the saved CSV column ordering (mean_diff = A - B)
# negative cohen_d → cond_B > cond_A
_CONTRASTS = [
    dict(
        name   = 'proleft_vs_antiright',
        label  = 'ProLeft > AntiRight (within Agreed)',
        csv    = CONTRAST_DIR / 'contrast_antiright_vs_proleft_subject_level.csv',
        cond_A = 'AntiRight',
        cond_B = 'ProLeft',
    ),
    dict(
        name   = 'proright_vs_antileft',
        label  = 'ProRight > AntiLeft (within Disagreed)',
        csv    = CONTRAST_DIR / 'contrast_antileft_vs_proright_subject_level.csv',
        cond_A = 'AntiLeft',
        cond_B = 'ProRight',
    ),
    dict(
        name   = 'agreed_vs_disagreed',
        label  = 'Agreed vs Disagreed',
        csv    = CONTRAST_DIR / 'contrast_agreed_vs_disagreed_subject_level.csv',
        cond_A = 'Agreed',
        cond_B = 'Disagreed',
    ),
]

contrast_all_rows = []

for spec in _CONTRASTS:
    df  = pd.read_csv(spec['csv'])
    sig = df[df['significant'] == True].copy()

    print(f"{'='*62}")
    print(f"C — {spec['label']}  |  {len(sig)} significant parcels")
    print('='*62)

    out_dir     = NS_CONTRAST_DIR / spec['name']
    parcels_dir = out_dir / 'parcels'
    out_dir.mkdir(exist_ok=True)
    parcels_dir.mkdir(exist_ok=True)

    rows, pids = [], []

    for _, row in sig.iterrows():
        pname = row['parcel_name']
        pid   = _name_to_id.get(pname)
        if pid is None:
            print(f'  WARNING: {pname} not in atlas — skipped')
            continue

        pids.append(pid)
        hemi, net, sub = _parse_parcel(pname)
        mni = _mni_centroid(pid)
        d   = row['cohen_d']

        rows.append({
            'contrast':    spec['label'],
            'parcel_name': pname,
            'hemisphere':  hemi,
            'network':     net,
            'subregion':   sub,
            'aal_region':  _top_aal(pid),
            'mni_x': mni[0], 'mni_y': mni[1], 'mni_z': mni[2],
            'cohen_d':   round(float(d), 4),
            'direction': f"{spec['cond_B']} > {spec['cond_A']}" if d < 0
                         else f"{spec['cond_A']} > {spec['cond_B']}",
            'p_fdr':     round(float(row['p_fdr']), 5),
        })
        _save_mask([pid], parcels_dir / f'{pname}.nii.gz')

    if not pids:
        print('  No significant parcels — skipping mask creation.\n')
        continue

    _save_mask(pids, out_dir / 'mask_combined.nii.gz')
    _save_effect_map({_name_to_id[r['parcel_name']]: r['cohen_d'] for r in rows},
                     out_dir / 'cohend_map.nii.gz')

    cdf = pd.DataFrame(rows)
    cdf.to_csv(out_dir / 'parcel_locations.csv', index=False)

    print(f"  mask_combined.nii.gz  ({len(pids)} parcels)")
    print(f"  cohend_map.nii.gz")
    print(f"  parcel_locations.csv")
    print()
    display(
        cdf[['parcel_name', 'network', 'hemisphere', 'subregion',
             'aal_region', 'mni_x', 'mni_y', 'mni_z', 'cohen_d', 'direction', 'p_fdr']]
        .sort_values('cohen_d')
        .reset_index(drop=True)
    )
    contrast_all_rows.extend(rows)

master = NS_CONTRAST_DIR / 'all_contrasts_parcel_locations.csv'
pd.DataFrame(contrast_all_rows).to_csv(master, index=False)
print(f'Master CSV ({len(contrast_all_rows)} rows) → {master.relative_to(ROOT)}')

---
## Section D — Decoder Results (contrast JSONs from neurosynth.org)

Reads the JSON files downloaded from neurosynth.org/decode/ for the 3 contrast masks  
and visualises the top/bottom `TOP_N` terms.

To generate the JSON files: upload each `mask_combined.nii.gz` from  
`contrasts/neurosynth/{contrast}/` to [neurosynth.org/decode/](https://neurosynth.org/decode/),  
download the result, and save it as `data/neurosynth/{contrast_name}.json`.

In [ ]:
_DECODE_CONTRASTS = [
    ('proleft_vs_antiright',  'ProLeft > AntiRight (within Agreed)'),
    ('proright_vs_antileft',  'ProRight > AntiLeft (within Disagreed)'),
    ('agreed_vs_disagreed',   'Agreed vs Disagreed'),
]

for cname, clabel in _DECODE_CONTRASTS:
    json_path = DECODER_DIR / f'{cname}.json'
    if not json_path.exists():
        print(f'[MISSING]  {json_path.relative_to(ROOT)}')
        continue

    with open(json_path) as _f:
        raw = json.load(_f)['data']['values']
    raw.pop('', None)

    s   = pd.Series(raw).sort_values(ascending=False)
    top = s.head(TOP_N)
    bot = s.tail(TOP_N)

    print(f'{'='*62}')
    print(f'  {clabel}')
    print(f'{'='*62}')

    tbl = pd.concat([
        top.rename('r').reset_index().rename(columns={'index': 'term'}).assign(direction='positive'),
        bot.sort_values().rename('r').reset_index().rename(columns={'index': 'term'}).assign(direction='negative'),
    ], ignore_index=True)
    display(tbl)

    fig, (ax_pos, ax_neg) = plt.subplots(1, 2, figsize=(14, 5))

    top_sorted = top.sort_values()
    ax_pos.barh(top_sorted.index, top_sorted.values, color='steelblue')
    ax_pos.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax_pos.set_xlabel('Correlation')
    ax_pos.set_title(f'Top {TOP_N} positive')

    bot_sorted = bot.sort_values(ascending=False)
    ax_neg.barh(bot_sorted.index, bot_sorted.values, color='tomato')
    ax_neg.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax_neg.set_xlabel('Correlation')
    ax_neg.set_title(f'Top {TOP_N} negative')

    fig.suptitle(clabel, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()